# Tarea 4 — Caso de Uso: Simulación de Campaña Comercial

**Objetivo:** Estimar cuánto dinero puede generar easyMoney con una campaña de cross-sell,  
combinando los resultados de la **Tarea 2** (propensión) y la **Tarea 3** (segmentación).  

**Preguntas de Carol:**
- ¿Cuánto dinero podemos ganar?
- ¿A cuántos clientes tenemos que impactar?
- ¿Qué productos ofrecer?
- ¿Impactamos a todos los segmentos?

**Framework:** Expected Value (Provost & Fawcett, 2013)  
`Beneficio = Σ(P(compra) × margen_producto) − (n_contactados × coste_contacto)`

---
## 1. Configuración e importación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

---
## 2. Carga de datos

Necesitamos:
1. **Probabilidades del modelo de propensión** (Tarea 2) — para cada cliente del test set
2. **Clusters asignados** (Tarea 3) — para segmentar la campaña
3. **Tabla de ventas y productos** — para calcular márgenes esperados

In [ ]:
# ============================================================
# 2.1 DATOS DEL MODELO DE PROPENSIÓN (Tarea 2)
# ============================================================
# INSTRUCCIÓN: Ejecuta primero el notebook de Feature Engineering
# y exporta las probabilidades con este código al final:
#
#   y_prob_test = pipe_final.predict_proba(X_test)[:, 1]
#   df_propension = X_test[['pk_partition']].copy()
#   df_propension['pk_cid'] = X_test.index
#   df_propension['prob_compra'] = y_prob_test
#   df_propension['y_real'] = y_test.values
#   # Quedarnos con el último mes disponible por cliente
#   df_propension = (
#       df_propension
#       .sort_values('pk_partition')
#       .groupby('pk_cid')
#       .last()
#       .reset_index()
#   )
#   df_propension.to_csv('propension_clientes.csv', index=False)
#
# Luego carga aquí:

df_prop = pd.read_csv('propension_clientes.csv')
print(f'Clientes con propensión: {len(df_prop):,}')
print(f'Probabilidad media: {df_prop["prob_compra"].mean():.4f}')
print(f'Compradores reales: {df_prop["y_real"].sum():,} ({df_prop["y_real"].mean()*100:.1f}%)')

In [ ]:
# ============================================================
# 2.2 DATOS DE SEGMENTACIÓN (Tarea 3)
# ============================================================
# INSTRUCCIÓN: Exporta del notebook de K-Means:
#
#   df_clusters = df_final_desescalado[['pk_cid', 'cluster']].copy()
#   df_clusters.to_csv('clientes_con_cluster.csv', index=False)
#
# Luego carga aquí:

df_clusters = pd.read_csv('clientes_con_cluster.csv')
print(f'Clientes con cluster: {len(df_clusters):,}')
print(f'\nDistribución:')
print(df_clusters['cluster'].value_counts().sort_index())

In [ ]:
# ============================================================
# 2.3 VENTAS Y PRODUCTOS
# ============================================================
df_sales = pd.read_csv('sales.csv', index_col=0)
df_prod  = pd.read_csv('product_description.csv', index_col=0)

df_ventas = df_sales.merge(df_prod, left_on='product_ID', right_on='pk_product_ID', how='left')

# Margen medio por producto
margen_producto = df_ventas.groupby('product_desc')['net_margin'].mean().round(2)
print('=== Margen medio por producto ===')
print(margen_producto.sort_values(ascending=False).to_string())

In [ ]:
# ============================================================
# 2.4 UNIÓN: propensión + cluster
# ============================================================
df = df_prop.merge(df_clusters, on='pk_cid', how='left')

print(f'Clientes con propensión Y cluster: {df["cluster"].notna().sum():,}')
print(f'Clientes sin cluster (no estaban en ventas históricas): {df["cluster"].isna().sum():,}')

# Los clientes sin cluster son los que nunca compraron → asignar cluster -1
df['cluster'] = df['cluster'].fillna(-1).astype(int)

---
## 3. Tabla de escenarios de campaña

El modelo de propensión asigna a cada cliente una probabilidad de compra.  
El **threshold** define a partir de qué probabilidad contactamos al cliente.  
Cada threshold produce un escenario diferente de coste y beneficio.

In [ ]:
# ============================================================
# SUPUESTOS DE COSTES
# ============================================================
# Definimos costes por tipo de canal. Carol puede ajustar estos valores.

COSTE_EMAIL = 0.50        # EUR por contacto (email marketing)
COSTE_PUSH  = 0.20        # EUR por contacto (push notification)
COSTE_CALL  = 5.00        # EUR por contacto (llamada comercial)

# Margen medio ponderado por volumen histórico de ventas
MARGEN_MEDIO = df_ventas['net_margin'].mean()  # ~607 EUR

# Margen mediano (más conservador, no distorsionado por pension_plan)
MARGEN_MEDIANO = df_ventas['net_margin'].median()  # ~69 EUR

print(f'Margen medio por venta:   {MARGEN_MEDIO:,.2f} EUR')
print(f'Margen mediano por venta: {MARGEN_MEDIANO:,.2f} EUR')
print(f'\nEl margen medio está inflado por pension_plan (5,976 EUR/venta, 8% del volumen).')
print(f'Usaremos AMBOS para dar un rango realista a Carol.')

In [ ]:
# ============================================================
# TABLA DE ESCENARIOS POR THRESHOLD
# ============================================================
escenarios = []

for t in np.arange(0.15, 0.65, 0.05):
    seleccionados = df[df['prob_compra'] >= t]
    n_contactados = len(seleccionados)
    compradores_reales = seleccionados['y_real'].sum()

    if n_contactados == 0:
        continue

    precision = compradores_reales / n_contactados
    recall = compradores_reales / df['y_real'].sum() if df['y_real'].sum() > 0 else 0

    # Ingresos: solo los que realmente compran generan margen
    ingreso_medio   = compradores_reales * MARGEN_MEDIO
    ingreso_mediano = compradores_reales * MARGEN_MEDIANO

    for canal, coste in [('Email', COSTE_EMAIL), ('Push', COSTE_PUSH), ('Llamada', COSTE_CALL)]:
        coste_total = n_contactados * coste
        escenarios.append({
            'threshold': round(t, 2),
            'canal': canal,
            'contactados': n_contactados,
            'compradores': int(compradores_reales),
            'precision': round(precision * 100, 1),
            'recall': round(recall * 100, 1),
            'ingreso_esperado': round(ingreso_medio, 0),
            'coste_campaña': round(coste_total, 0),
            'beneficio_neto': round(ingreso_medio - coste_total, 0),
            'roi_pct': round((ingreso_medio - coste_total) / coste_total * 100, 0) if coste_total > 0 else 0
        })

df_escenarios = pd.DataFrame(escenarios)

# Mostrar solo canal Email para no saturar
print('=== ESCENARIOS DE CAMPAÑA (Canal: Email, coste 0.50 EUR/contacto) ===')
print(f'Margen por comprador: {MARGEN_MEDIO:,.2f} EUR (medio ponderado)\n')
cols_show = ['threshold','contactados','compradores','precision','recall',
             'ingreso_esperado','coste_campaña','beneficio_neto','roi_pct']
print(df_escenarios[df_escenarios['canal']=='Email'][cols_show].to_string(index=False))

---
## 4. Visualización del trade-off

In [ ]:
df_email = df_escenarios[df_escenarios['canal'] == 'Email'].copy()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- Gráfico 1: Beneficio neto por threshold ---
ax = axes[0]
colors = ['#2c7fb8' if b > 0 else '#d95f0e' for b in df_email['beneficio_neto']]
ax.bar(df_email['threshold'].astype(str), df_email['beneficio_neto'], color=colors, edgecolor='white')
ax.set_xlabel('Threshold')
ax.set_ylabel('Beneficio neto (EUR)')
ax.set_title('Beneficio neto por threshold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.axhline(0, color='black', linewidth=0.8)

# --- Gráfico 2: Precision vs Recall ---
ax = axes[1]
ax.plot(df_email['threshold'], df_email['precision'], 'o-', color='#2c7fb8', label='Precisión (%)', linewidth=2)
ax.plot(df_email['threshold'], df_email['recall'], 's-', color='#d95f0e', label='Recall (%)', linewidth=2)
ax.set_xlabel('Threshold')
ax.set_ylabel('%')
ax.set_title('Precisión vs Recall')
ax.legend()
ax.grid(alpha=0.3)

# --- Gráfico 3: Contactados vs Compradores detectados ---
ax = axes[2]
ax.bar(df_email['threshold'].astype(str), df_email['contactados'],
       color='#7fcdbb', edgecolor='white', label='Contactados')
ax.bar(df_email['threshold'].astype(str), df_email['compradores'],
       color='#2c7fb8', edgecolor='white', label='Compradores reales')
ax.set_xlabel('Threshold')
ax.set_ylabel('Clientes')
ax.set_title('Alcance de la campaña')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Análisis de escenarios de campaña — Canal Email', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Comparativa por canal de contacto

In [ ]:
# Mejor threshold (max F1) = 0.45 según Tarea 2
THRESHOLD_OPTIMO = 0.45

print(f'=== Comparativa de canales con threshold = {THRESHOLD_OPTIMO} ===\n')

df_canales = df_escenarios[df_escenarios['threshold'] == THRESHOLD_OPTIMO].copy()

for _, row in df_canales.iterrows():
    print(f"Canal: {row['canal']}")
    print(f"  Contactados:      {row['contactados']:>10,}")
    print(f"  Compradores:      {row['compradores']:>10,}")
    print(f"  Ingreso esperado: {row['ingreso_esperado']:>10,.0f} EUR")
    print(f"  Coste campaña:    {row['coste_campaña']:>10,.0f} EUR")
    print(f"  Beneficio neto:   {row['beneficio_neto']:>10,.0f} EUR")
    print(f"  ROI:              {row['roi_pct']:>10,.0f}%")
    print()

---
## 6. Análisis por segmento (clusters)

¿A qué clusters deberíamos dirigir la campaña?  
Cruzamos la probabilidad de compra con el perfil de cada segmento.

In [ ]:
# Análisis por cluster
THRESHOLD = THRESHOLD_OPTIMO

df['seleccionado'] = (df['prob_compra'] >= THRESHOLD).astype(int)
df['acierto'] = ((df['seleccionado'] == 1) & (df['y_real'] == 1)).astype(int)

# Solo clusters válidos (excluir -1 si hay muchos sin cluster)
df_con_cluster = df[df['cluster'] >= 0].copy()

perfil_cluster = df_con_cluster.groupby('cluster').agg(
    total_clientes=('pk_cid', 'count'),
    prob_media=('prob_compra', 'mean'),
    contactados=('seleccionado', 'sum'),
    compradores_reales=('y_real', 'sum'),
    aciertos=('acierto', 'sum'),
).round(4)

perfil_cluster['tasa_compra_real'] = (perfil_cluster['compradores_reales'] / perfil_cluster['total_clientes'] * 100).round(1)
perfil_cluster['precision'] = (perfil_cluster['aciertos'] / perfil_cluster['contactados'] * 100).round(1)
perfil_cluster['ingreso_estimado'] = (perfil_cluster['aciertos'] * MARGEN_MEDIO).round(0)
perfil_cluster['coste_email'] = (perfil_cluster['contactados'] * COSTE_EMAIL).round(0)
perfil_cluster['beneficio_neto'] = perfil_cluster['ingreso_estimado'] - perfil_cluster['coste_email']

print('=== OPORTUNIDAD POR SEGMENTO ===')
print(f'Threshold: {THRESHOLD} | Canal: Email ({COSTE_EMAIL} EUR/contacto)\n')
print(perfil_cluster[['total_clientes','tasa_compra_real','contactados','aciertos',
                       'precision','ingreso_estimado','coste_email','beneficio_neto']].to_string())

In [ ]:
# Visualización: beneficio por cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Beneficio neto por cluster ---
ax = axes[0]
clusters_sorted = perfil_cluster.sort_values('beneficio_neto', ascending=True)
colors = ['#2c7fb8' if b > 0 else '#d95f0e' for b in clusters_sorted['beneficio_neto']]
bars = ax.barh([f'C{c}' for c in clusters_sorted.index], clusters_sorted['beneficio_neto'],
               color=colors, edgecolor='white')
ax.set_xlabel('Beneficio neto (EUR)')
ax.set_title('Beneficio neto estimado por cluster')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.axvline(0, color='black', linewidth=0.8)

for bar, val in zip(bars, clusters_sorted['beneficio_neto']):
    ax.text(bar.get_width() + (500 if val >= 0 else -500),
            bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}€', va='center', fontsize=9,
            ha='left' if val >= 0 else 'right')

# --- Tasa de compra real vs precisión del modelo ---
ax = axes[1]
x = np.arange(len(perfil_cluster))
width = 0.35
ax.bar(x - width/2, perfil_cluster['tasa_compra_real'], width,
       label='Tasa compra real (%)', color='#7fcdbb', edgecolor='white')
ax.bar(x + width/2, perfil_cluster['precision'], width,
       label='Precisión modelo (%)', color='#2c7fb8', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f'C{c}' for c in perfil_cluster.index])
ax.set_ylabel('%')
ax.set_title('Tasa de compra real vs precisión del modelo')
ax.legend()

plt.suptitle('Análisis de oportunidad por segmento', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. ¿Qué productos ofrecer a cada segmento?

Cruzamos el perfil de tenencia de cada cluster con los productos que generan más margen  
para identificar la **oportunidad de cross-sell** por segmento.

In [ ]:
# ── Tamaños reales de cada cluster desde clientes_con_cluster.csv ────────────
n_por_cluster = df_clusters['cluster'].value_counts().sort_index().to_dict()

# ── Definición de estrategia por cluster ─────────────────────────────────────
# n_clientes se obtiene directamente de clientes_con_cluster.csv;
# el resto de campos (accion, producto, margen) reflejan el análisis de la Tarea 3.

estrategia_def = [
    {
        'cluster': 0,
        'nombre': 'Cliente Joven en Riesgo',
        'accion': 'Reactivación',
        'producto_recomendado': 'pension_plan / short_term_deposit',
        'margen_esperado': 5976.44,
        'justificacion': 'Perfil joven con ingresos medios, bajo engagement. '
                         'Producto de inversión a largo plazo para capturar ahorro temprano.'
    },
    {
        'cluster': 1,
        'nombre': 'Masa Inactiva Básica',
        'accion': 'Activación cross-sell',
        'producto_recomendado': 'debit_card / payroll',
        'margen_esperado': 60.00,
        'justificacion': 'El cluster más grande. Solo tienen cuenta. '
                         'Primer paso: vincular con tarjeta débito o nómina para aumentar engagement.'
    },
    {
        'cluster': 2,
        'nombre': 'Cliente Leal Operativo',
        'accion': 'Cross-sell inversión',
        'producto_recomendado': 'funds / pension_plan',
        'margen_esperado': 1499.05,
        'justificacion': 'Alta operatividad pero sin productos de inversión. '
                         'Camino natural hacia el perfil C3 (VIP). Mayor potencial de margen.'
    },
    {
        'cluster': 3,
        'nombre': 'Cliente VIP Multiproducto',
        'accion': 'Retención',
        'producto_recomendado': 'Productos exclusivos / upgrades',
        'margen_esperado': 5108.00,
        'justificacion': 'Los más rentables. No impactar con cross-sell agresivo. '
                         'Foco en retención y servicio personalizado.'
    },
    {
        'cluster': 4,
        'nombre': 'Cliente Maduro Dormido',
        'accion': 'Reactivación',
        'producto_recomendado': 'payroll_account / credit_card',
        'margen_esperado': 70.00,
        'justificacion': 'Alta antigüedad pero bajo engagement. '
                         'Incentivar uso con productos de operatividad.'
    },
    {
        'cluster': 5,
        'nombre': 'Ultra-HNW / Outlier',
        'accion': 'Estrategia ad hoc',
        'producto_recomendado': 'Gestión personalizada',
        'margen_esperado': 840.00,
        'justificacion': 'Grupo demasiado pequeño para campaña masiva. '
                         'Gestión individual por el equipo comercial.'
    },
    {
        'cluster': 6,
        'nombre': 'Cliente Premium Financiado',
        'accion': 'Retención + up-sell',
        'producto_recomendado': 'funds / securities',
        'margen_esperado': 1497.83,
        'justificacion': 'Único cluster con financiación. '
                         'Cross-sell hacia inversión para migrar a perfil C3.'
    },
]

# Inyectar n_clientes real desde clientes_con_cluster.csv
for row in estrategia_def:
    row['n_clientes'] = n_por_cluster.get(row['cluster'], 0)

estrategia = pd.DataFrame(estrategia_def)

print('=== ESTRATEGIA COMERCIAL POR SEGMENTO ===')
print(f'(n_clientes cargado desde clientes_con_cluster.csv — {len(df_clusters):,} clientes totales)\n')
for _, row in estrategia.iterrows():
    print(f"C{row['cluster']} — {row['nombre']} ({row['n_clientes']:,} clientes)")
    print(f"  Acción:       {row['accion']}")
    print(f"  Producto:     {row['producto_recomendado']}")
    print(f"  Margen/venta: {row['margen_esperado']:,.2f} EUR")
    print(f"  → {row['justificacion']}")
    print()

# ── Exportar estrategia enriquecida a clientes_con_cluster.csv ───────────────
estrategia.to_csv('clientes_con_cluster.csv', index=False)
print('✓ Estrategia exportada a clientes_con_cluster.csv')


---
## 8. Estimación de ingresos de la campaña

Combinamos:
- **Probabilidad de compra** del modelo (por cliente)
- **Margen esperado** por producto recomendado (por cluster)
- **Coste de contacto** por canal

Fórmula del Expected Value Framework:  
`Beneficio = Σ P(compra_i) × margen_producto_j − n_contactados × coste_contacto`

In [ ]:
# Asignar margen esperado por cluster
margen_map = estrategia.set_index('cluster')['margen_esperado'].to_dict()
accion_map = estrategia.set_index('cluster')['accion'].to_dict()

df_con_cluster['margen_esperado'] = df_con_cluster['cluster'].map(margen_map)
df_con_cluster['accion'] = df_con_cluster['cluster'].map(accion_map)

# Valor esperado por cliente = prob_compra × margen del producto recomendado
df_con_cluster['valor_esperado'] = df_con_cluster['prob_compra'] * df_con_cluster['margen_esperado']

# Seleccionar clientes según threshold
campaña = df_con_cluster[df_con_cluster['prob_compra'] >= THRESHOLD_OPTIMO].copy()

print(f'=== ESTIMACIÓN DE CAMPAÑA (threshold = {THRESHOLD_OPTIMO}) ===\n')

resumen = campaña.groupby('cluster').agg(
    n_contactar=('pk_cid', 'count'),
    prob_media=('prob_compra', 'mean'),
    valor_esperado_total=('valor_esperado', 'sum'),
    compradores_reales=('y_real', 'sum'),
).round(2)

resumen['coste_email'] = resumen['n_contactar'] * COSTE_EMAIL
resumen['beneficio_neto'] = resumen['valor_esperado_total'] - resumen['coste_email']

# Añadir nombre del cluster
nombre_map = estrategia.set_index('cluster')['nombre'].to_dict()
resumen['segmento'] = resumen.index.map(nombre_map)

print(resumen[['segmento','n_contactar','prob_media','valor_esperado_total',
               'coste_email','beneficio_neto','compradores_reales']].to_string())

print(f'\n{"="*60}')
print(f'TOTAL CLIENTES A CONTACTAR:  {resumen["n_contactar"].sum():>10,}')
print(f'INGRESO ESPERADO TOTAL:      {resumen["valor_esperado_total"].sum():>10,.0f} EUR')
print(f'COSTE TOTAL (Email):         {resumen["coste_email"].sum():>10,.0f} EUR')
print(f'BENEFICIO NETO ESTIMADO:     {resumen["beneficio_neto"].sum():>10,.0f} EUR')
print(f'{"="*60}')

In [ ]:
# Visualización final
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Beneficio por cluster ---
ax = axes[0]
resumen_sorted = resumen.sort_values('beneficio_neto', ascending=True)
colors_bar = ['#2c7fb8' if b > 0 else '#d95f0e' for b in resumen_sorted['beneficio_neto']]
bars = ax.barh([f"C{c}\n{resumen_sorted.loc[c,'segmento']}" for c in resumen_sorted.index],
               resumen_sorted['beneficio_neto'], color=colors_bar, edgecolor='white')
ax.set_xlabel('Beneficio neto estimado (EUR)')
ax.set_title('Beneficio neto por segmento (Expected Value)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.axvline(0, color='black', linewidth=0.8)

# --- Composición del ingreso ---
ax = axes[1]
labels = [f"C{c}" for c in resumen.sort_values('valor_esperado_total', ascending=False).index]
sizes = resumen.sort_values('valor_esperado_total', ascending=False)['valor_esperado_total']
colors_pie = plt.cm.Set2.colors[:len(resumen)]
wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                   colors=colors_pie, startangle=90)
ax.set_title('Distribución del ingreso esperado por segmento')

plt.suptitle('Resultado de la campaña por segmento', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Recomendación final para Carol

### Resumen ejecutivo

Utilizando el modelo de propensión (Tarea 2) y la segmentación de clientes (Tarea 3),
hemos diseñado una campaña comercial con los siguientes resultados esperados:

| Concepto | Valor |
|----------|-------|
| **Clientes impactables** | _Rellenar con resultados_ |
| **Tasa de conversión esperada** | _Rellenar con resultados_ |
| **Ingreso esperado** | _Rellenar con resultados_ |
| **Coste de la campaña (Email)** | _Rellenar con resultados_ |
| **Beneficio neto** | _Rellenar con resultados_ |
| **ROI** | _Rellenar con resultados_ |

### Prioridades estratégicas

| Prioridad | Segmento | Acción | Producto | Por qué |
|-----------|----------|--------|----------|---------|
| 1 | C2 — Operativo | Cross-sell inversión | Fondos / Pension Plan | Mayor potencial de migración a VIP |
| 2 | C6 — Premium Financiado | Up-sell inversión | Securities / Fondos | Ya diversificados, falta inversión |
| 3 | C1 — Monoproducto | Activación | Tarjeta débito / Nómina | Volumen grande, margen bajo pero engagement |
| 4 | C4 — Maduro Pasivo | Reactivación | Cuenta nómina / Credit card | Antigüedad alta, necesitan incentivo |
| 5 | C3 / C6 — VIP | Retención | Gestión personalizada | No arriesgar con campañas genéricas |

### Riesgos y limitaciones

1. **El modelo predice compra genérica**, no por producto específico — el margen real
   dependerá de qué producto compre cada cliente, no del que le recomendamos.
2. **Los costes de contacto son estimados** — Carol debe ajustarlos con datos reales.
3. **Sin datos de comportamiento digital** (uso de app, clics), el modelo tiene un
   techo de recall (~52%) — invertir en captura de datos de interacción mejoraría
   significativamente los resultados futuros.

### Próximos pasos

1. Validar los costes de contacto con el equipo de Marketing Directo (Erin)
2. Ejecutar un **test A/B** con el 10% de los clientes seleccionados antes del lanzamiento masivo
3. Medir resultados reales y recalibrar el modelo trimestralmente
4. Proponer a IT la captura de datos de interacción digital para mejorar el modelo